In [2]:
import os
import sys

# 手动指定你 legato 项目的根目录路径
# 根据你之前提供的信息，路径应该是：
PROJECT_ROOT = r"C:/Users/20810/Desktop/Code/legato"

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import legato
print("Legato 模块加载成功！")

Legato 模块加载成功！


In [ ]:
import sys
from pathlib import Path

import torch
from PIL import Image
from transformers import AutoProcessor, GenerationConfig

# 自动定位项目根目录，避免依赖单元执行顺序
_cwd = Path.cwd().resolve()
if (_cwd / "legato").exists() and (_cwd / "models").exists():
    PROJECT_ROOT = _cwd
elif (_cwd.parent / "legato").exists() and (_cwd.parent / "models").exists():
    PROJECT_ROOT = _cwd.parent
else:
    PROJECT_ROOT = Path(r"C:/Users/20810/Desktop/Code/legato")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from legato.models import LegatoModel

snapshot_root = PROJECT_ROOT / "models" / "models--guangyangmusic--legato" / "snapshots"
snapshot_dirs = sorted([p for p in snapshot_root.iterdir() if p.is_dir()])
if not snapshot_dirs:
    raise FileNotFoundError(f"No local model snapshot found under: {snapshot_root}")

model_path = snapshot_dirs[0]
model = LegatoModel.from_pretrained(str(model_path), local_files_only=True)
print(model.config._name_or_path)

c:\Users\20810\.conda\envs\guangyang\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(
c:\Users\20810\.conda\envs\guangyang\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'legato'

In [13]:
model = model.to("cuda").half()
processor = AutoProcessor.from_pretrained("guangyangmusic/legato")

In [6]:
import torch
from PIL import Image
from transformers import AutoProcessor, GenerationConfig
from legato.models import LegatoModel

# Load model and processor
model = LegatoModel.from_pretrained("guangyangmusic/legato")
model = model.to("cuda").half()  # Use FP16
processor = AutoProcessor.from_pretrained("guangyangmusic/legato")

# Move to GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

# Load and process image
image = Image.open("./images/image (1).jpg").convert("RGB")
inputs = processor(images=image, return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}

# Generate ABC notation
generation_config = GenerationConfig(
    max_length=2048,
    num_beams=10,
    repetition_penalty=1.1,
    pad_token_id=processor.tokenizer.pad_token_id,
    eos_token_id=processor.tokenizer.eos_token_id
)

with torch.no_grad():
    outputs = model.generate(**inputs, generation_config=generation_config)

# Decode output
abc_notation = processor.batch_decode(outputs, skip_special_tokens=True)[0]
print(abc_notation)

Loading checkpoint shards: 100%|██████████| 5/5 [00:07<00:00,  1.49s/it]
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
`generation_config` default values have been modified to match model-specific defaults: {'bos_token_id': 1, 'load_pretrained_encoder': False}. If this is not desired, please set these values explicitly.


X:1
T:<|text|>
L:1/8
M:4/4
I:linebreak $
K:G
V:1 treble
V:1
 G2 FG E2 DE | G2 BG DGBG | G2 FG E2 GA | Bded B2 AF | G2 FG E2 DE | %5
 G2 BG DGBG |$ G2 FG E2 GA | Bded B2 Bc || dBGB d2 Bc | dBGB e2 dc | %10
 dBGB d2 Bd | eged B2 Bc |$ dBGB d2 Bd | g2 b2 egdB | G3 B AGAB | %15
 dfgd B2 AF || %16



In [15]:
# Move to GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

In [16]:
# Load and process image
image = Image.open("./images/image2.jpg").convert("RGB")
inputs = processor(images=image, return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}

# Generate ABC notation
generation_config = GenerationConfig(
    max_length=2048,
    num_beams=10,
    repetition_penalty=1.1,
    pad_token_id=processor.tokenizer.pad_token_id,
    eos_token_id=processor.tokenizer.eos_token_id
)

with torch.no_grad():
    outputs = model.generate(**inputs, generation_config=generation_config)

# Decode output
abc_notation = processor.batch_decode(outputs, skip_special_tokens=True)[0]
print(abc_notation)

`generation_config` default values have been modified to match model-specific defaults: {'bos_token_id': 1, 'load_pretrained_encoder': False}. If this is not desired, please set these values explicitly.


X:1
%%score { 1 | 2 }
L:1/8
M:3/4
I:linebreak $
K:D
V:1 treble
V:2 bass
V:1
 z2 z2 A2 | A4 A2 | A4 ^G2 | [DG]4 [CG]2 |[M:4/4] [DF]6 [DA]2 | %5
[M:3/4] [Fd]4 d2 | d4 e2 | c6 |[M:1/4] [Ac]2 |[M:3/4] [Ad]4 [Ad]2 | %10
 c4 B2 |$ A4 ^G2 |[M:4/4] [CG]6 F2 |[M:3/4] F4 E2 | B4 C2 | %15
 D6 |] %16
V:2
 z2 z2 [F,A,D]2 | [F,A,D]4 [F,A,D]2 | [E,B,D]4 [E,B,D]2 | [A,,A,]4 [A,,A,]2 |[M:4/4] [D,A,]6 [F,A,]2 | %5
[M:3/4] [B,,B,]4[K:treble] [B,DF]2 | [B,D^G]4 [B,DG]2 | [A,EA]6 |[M:1/4] [G,E]2 |[M:3/4][K:bass] [F,D]4 [F,D]2 | %10
 [G,D]4 [G,D]2 |$ [F,A,D]4 [E,B,D]2 |[M:4/4] [A,,A,]6 [D,A,D]2 |[M:3/4] [G,B,]4 [G,B,]2 | [G,B,E]4 [A,,G,A,]2 | %15
 [D,F,A,]6 |] %16

